# HardPair-Net-Lite — HAM10000 experiment

This notebook implements `Methodology.txt` cell by cell. Use the **Python (histo)** kernel. The two long fine-tuning cells save atomic checkpoints every hour and at every epoch boundary; rerunning an interrupted cell resumes at the saved epoch and sample offset. Feature extraction is resumable too.

The final test split is not accessed until calibration, gate fitting, and checkpoint selection are complete.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import copy, json, platform, sys

PROJECT_ROOT = Path.cwd().resolve()
PART_DIR = PROJECT_ROOT / 'experiment_parts' / 'part_01_ham10000_core_experiment'
if not PART_DIR.is_dir():
    PART_DIR = Path.cwd().resolve()
sys.path.insert(0, str(PART_DIR / 'code'))
import pandas as pd
import torch

from hardpair.checkpoint import atomic_torch_save
from hardpair.config import load_config, resolve_device, seed_everything
from hardpair.data import MEL_IDX, NV_IDX, build_manifest, create_split_manifest, split_summary
from hardpair.evaluation import evaluate_probabilities, paired_group_bootstrap
from hardpair.fusion import apply_gate, fit_temperature, pair_restricted_fusion, train_gate
from hardpair.model import HardPairNet
from hardpair.training import (benchmark_training_steps, extract_feature_cache, initialize_heads, predict_model, train_image_model)

c:\Users\Bilal Rehman\miniconda3\envs\histo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Configuration and reproducibility
config = load_config(PART_DIR / 'configuration' / 'config.yaml')
config['_smoke'] = False
config['training']['checkpoint_interval_seconds'] = 3600  # one hour
assert config['training']['max_finetune_epochs'] == 20
assert config['training']['min_finetune_epochs'] == 20  # force completion of all 20 epochs
seed_everything(config['seed'])
device = resolve_device(config)
run_dir = Path(config['paths']['output_dir']) / config['paths'].get('full_run_dir', 'full')
run_dir.mkdir(parents=True, exist_ok=True)
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'device': str(device)})
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

{'python': '3.10.20', 'torch': '2.2.2+cu118', 'cuda': '11.8', 'device': 'cuda'}
NVIDIA GeForce GTX 1050


## 1. Validate HAM10000 and freeze the lesion-disjoint split

The split file is created once and reused. Seven folds form training; one fold each is reserved for fusion fitting, validation, and the untouched test set.

In [3]:
manifest = build_manifest(config, verify_images=False)
split_path = Path(config['paths']['output_dir']) / 'ham10000_splits.csv'
if split_path.exists():
    splits = pd.read_csv(split_path)
    assert splits.image_id.tolist() == manifest.image_id.tolist(), 'Existing split does not match metadata order'
else:
    splits = create_split_manifest(manifest, config, split_path)
assert len(splits) == 10015
assert splits.image_id.nunique() == 10015
assert splits.groupby('lesion_id').split.nunique().max() == 1
summary = split_summary(splits)
summary.to_csv(run_dir / 'split_summary.csv')
display(summary)

dx,akiec,bcc,bkl,df,mel,nv,vasc,All
split,,,,,,,,
fusion,38,66,101,10,111,651,15,992
test,35,55,93,4,125,675,23,1010
train,212,340,792,96,759,4694,92,6985
val,42,53,113,5,118,685,12,1028
All,327,514,1099,115,1113,6705,142,10015


## 2. Verify the controlled-fusion mathematics

In [4]:
torch.manual_seed(7)
p_check = torch.softmax(torch.randn(32, 7), 1)
s_check = torch.softmax(torch.randn(32, 2), 1)
a_check = 0.75 * torch.rand(32, 1)
q_check = pair_restricted_fusion(p_check, s_check, a_check)
offpair = [i for i in range(7) if i not in (MEL_IDX, NV_IDX)]
assert (q_check[:, offpair] - p_check[:, offpair]).abs().max() < 1e-6
assert (q_check[:, [MEL_IDX, NV_IDX]].sum(1) - p_check[:, [MEL_IDX, NV_IDX]].sum(1)).abs().max() < 1e-6
assert (q_check.sum(1) - 1).abs().max() < 1e-6
print('Fusion invariants passed.')

Fusion invariants passed.


## 3. Cache deterministic pretrained features and initialize heads

The feature cache uses an hourly progress file. The head stage is inexpensive and writes a checkpoint at each epoch.

In [5]:
pretrained = HardPairNet(pretrained=True)
feature_cache = extract_feature_cache(pretrained, splits, run_dir / 'pretrained_features.pt', config, device)
# Compatibility guard for caches produced before the inference-tensor fix.
feature_cache['features'] = feature_cache['features'].detach().clone()
assert not feature_cache['features'].is_inference()
head_initialization = initialize_heads(pretrained, feature_cache, run_dir / 'head_initialization.pt', config, device)
print({k: v for k, v in head_initialization.items() if k not in ('model_state',)})

{'complete': True, 'generalist_validation_macro_f1': 0.5461626691034281, 'generalist_validation_mel_sensitivity': 0.4152542372881356, 'specialist_validation_accuracy': 0.8978829383850098}


## 4. Profile 100 training steps

Run this once before the full fine-tuning cells. It reports the projected duration of one complete 20-epoch run on the current GPU. Both ablations use the same 20-epoch schedule.

In [6]:
train_frame = splits.loc[splits.split == 'train'].reset_index(drop=True)
val_frame = splits.loc[splits.split == 'val'].reset_index(drop=True)
profile_model = HardPairNet(pretrained=True)
profile_model.load_state_dict(head_initialization['model_state'])
profile = benchmark_training_steps(profile_model, train_frame, config, device, steps=100, warmup=10)
steps_per_epoch = -(-len(train_frame) // config['training']['batch_size'])
profile['projected_hours_per_run'] = profile['seconds_per_step'] * steps_per_epoch * config['training']['max_finetune_epochs'] / 3600
with (run_dir / 'hardware_profile.json').open('w', encoding='utf-8') as handle:
    json.dump(profile, handle, indent=2)
profile

{'device': 'cuda',
 'steps': 100.0,
 'seconds_per_step': 0.02011858900252264,
 'peak_allocated_gb': 0.09145736694335938,
 'projected_hours_per_run': 0.09768692660113772}

## 5. Experiment A — seven-class baseline

Safe to rerun after interruption: `baseline_latest.pt` resumes mid-epoch from the last hourly checkpoint.

In [7]:
baseline = HardPairNet(pretrained=True)
baseline.load_state_dict(head_initialization['model_state'])
baseline, baseline_training = train_image_model(
    baseline, train_frame, val_frame, 'baseline', run_dir, config, device, multitask=False
)
baseline_training['history']

[{'epoch': 1,
  'training_loss': 0.7646624181918272,
  'validation_macro_f1': 0.4745405134769318,
  'validation_mel_sensitivity': 0.3644067796610169},
 {'epoch': 2,
  'training_loss': 0.712011170884329,
  'validation_macro_f1': 0.534770292942326,
  'validation_mel_sensitivity': 0.4322033898305085},
 {'epoch': 3,
  'training_loss': 0.6768135736105095,
  'validation_macro_f1': 0.5103566935992017,
  'validation_mel_sensitivity': 0.3728813559322034},
 {'epoch': 4,
  'training_loss': 0.6455810904822865,
  'validation_macro_f1': 0.5049599852529256,
  'validation_mel_sensitivity': 0.288135593220339},
 {'epoch': 5,
  'training_loss': 0.6302063850457956,
  'validation_macro_f1': 0.5291622729874569,
  'validation_mel_sensitivity': 0.4067796610169492},
 {'epoch': 6,
  'training_loss': 0.5985098487619573,
  'validation_macro_f1': 0.5275460126742625,
  'validation_mel_sensitivity': 0.3389830508474576},
 {'epoch': 7,
  'training_loss': 0.5854850335277313,
  'validation_macro_f1': 0.49765175487780666

## 6. Experiment B — multitask generalist + MEL/NV specialist

This starts from the same pretrained encoder and initialized heads as Experiment A. It has its own resumable checkpoint.

In [8]:
multitask = HardPairNet(pretrained=True)
multitask.load_state_dict(head_initialization['model_state'])
multitask, multitask_training = train_image_model(
    multitask, train_frame, val_frame, 'multitask', run_dir, config, device, multitask=True
)
multitask_training['history']

[{'epoch': 1,
  'training_loss': 0.8386115023946294,
  'validation_macro_f1': 0.4578888901033629,
  'validation_mel_sensitivity': 0.3389830508474576},
 {'epoch': 2,
  'training_loss': 0.7807919534119266,
  'validation_macro_f1': 0.5392154084959394,
  'validation_mel_sensitivity': 0.4406779661016949},
 {'epoch': 3,
  'training_loss': 0.7429863543716995,
  'validation_macro_f1': 0.5119999760154351,
  'validation_mel_sensitivity': 0.3644067796610169},
 {'epoch': 4,
  'training_loss': 0.7112873731294358,
  'validation_macro_f1': 0.5055343578643907,
  'validation_mel_sensitivity': 0.288135593220339},
 {'epoch': 5,
  'training_loss': 0.6925109492064205,
  'validation_macro_f1': 0.537580855949295,
  'validation_mel_sensitivity': 0.4067796610169492},
 {'epoch': 6,
  'training_loss': 0.6599939526095274,
  'validation_macro_f1': 0.5286444934746543,
  'validation_mel_sensitivity': 0.3474576271186441},
 {'epoch': 7,
  'training_loss': 0.6437841865384075,
  'validation_macro_f1': 0.4791345591450586

## 7. Freeze models and cache fusion/validation/test predictions

In [9]:
prediction_path = run_dir / 'cached_predictions.pt'
frames = {name: splits.loc[splits.split == name].reset_index(drop=True) for name in ('fusion', 'val', 'test')}
if prediction_path.exists():
    predictions = torch.load(prediction_path, map_location='cpu')
else:
    predictions = {'baseline': {}, 'multitask': {}}
    for model_name, model in (('baseline', baseline), ('multitask', multitask)):
        for split_name, frame in frames.items():
            predictions[model_name][split_name] = predict_model(model, frame, config, device)
    atomic_torch_save(predictions, prediction_path)
{model: {split: len(values['labels']) for split, values in split_values.items()} for model, split_values in predictions.items()}

{'baseline': {'fusion': 992, 'val': 1028, 'test': 1010},
 'multitask': {'fusion': 992, 'val': 1028, 'test': 1010}}

## 8. Fit temperatures on fusion data only

In [10]:
def fit_model_temperatures(output):
    labels = output['labels'].to(device)
    t7 = fit_temperature(output['generalist_logits'].to(device), labels)
    pair = (labels == MEL_IDX) | (labels == NV_IDX)
    t2 = fit_temperature(output['specialist_logits'].to(device)[pair], (labels[pair] == NV_IDX).long())
    return t7, t2

base_t7, base_t2 = fit_model_temperatures(predictions['baseline']['fusion'])
multi_t7, multi_t2 = fit_model_temperatures(predictions['multitask']['fusion'])
def calibrated(output, t7, t2):
    return (torch.softmax(output['generalist_logits'].float() / t7, 1),
            torch.softmax(output['specialist_logits'].float() / t2, 1))
cal = {'baseline': {}, 'multitask': {}}
for name in frames:
    cal['baseline'][name] = calibrated(predictions['baseline'][name], base_t7, base_t2)
    cal['multitask'][name] = calibrated(predictions['multitask'][name], multi_t7, multi_t2)
{'baseline_generalist': base_t7, 'multitask_generalist': multi_t7, 'specialist': multi_t2}

{'baseline_generalist': 1.5191583633422852,
 'multitask_generalist': 1.5054196119308472,
 'specialist': 1.7353448867797852}

## 9. Fit the learned gate on fusion data, selecting on validation data

In [11]:
fit_p, fit_s = cal['multitask']['fusion']
val_p, val_s = cal['multitask']['val']
gate, gate_stats = train_gate(
    fit_p, fit_s, predictions['multitask']['fusion']['labels'],
    val_p, val_s, predictions['multitask']['val']['labels'],
    config, device
)
atomic_torch_save({'gate_state': gate.state_dict(), **gate_stats}, run_dir / 'learned_gate.pt')
gate_stats['validation_loss']

[0.6170471906661987,
 0.6176013350486755,
 0.6180446147918701,
 0.6184436082839966,
 0.618826150894165]

## 10. One-time held-out test evaluation and lesion-level bootstrap

In [12]:
test_y = predictions['multitask']['test']['labels']
baseline_p, _ = cal['baseline']['test']
auxiliary_p, specialist_s = cal['multitask']['test']
fixed_p = pair_restricted_fusion(auxiliary_p, specialist_s, config['fusion']['fixed_alpha'])
learned_p, alpha = apply_gate(gate, gate_stats, auxiliary_p, specialist_s)
results = {
    'baseline': evaluate_probabilities(baseline_p, test_y),
    'auxiliary_only': evaluate_probabilities(auxiliary_p, test_y),
    'fixed_fusion': evaluate_probabilities(fixed_p, test_y, auxiliary_p),
    'learned_fusion': evaluate_probabilities(learned_p, test_y, auxiliary_p),
}
lesions = predictions['multitask']['test']['lesion_ids']
n_boot = config['evaluation']['bootstrap_resamples']
results['fixed_vs_auxiliary_bootstrap'] = paired_group_bootstrap(fixed_p, auxiliary_p, test_y, lesions, n_boot, config['seed'])
results['learned_vs_auxiliary_bootstrap'] = paired_group_bootstrap(learned_p, auxiliary_p, test_y, lesions, n_boot, config['seed'] + 1)
results['run'] = {'test_images': len(test_y), 'mean_alpha': float(alpha.mean()), 'device': str(device)}
with (run_dir / 'metrics.json').open('w', encoding='utf-8') as handle:
    json.dump(results, handle, indent=2, allow_nan=True)
atomic_torch_save({'labels': test_y, 'baseline': baseline_p, 'auxiliary_only': auxiliary_p, 'fixed_fusion': fixed_p, 'learned_fusion': learned_p, 'alpha': alpha}, run_dir / 'test_probabilities.pt')
model_names = ['baseline', 'auxiliary_only', 'fixed_fusion', 'learned_fusion']
common_metric_names = [
    'pair_balanced_accuracy', 'mel_sensitivity', 'mel_to_nv', 'nv_to_mel',
    'macro_f1', 'balanced_accuracy', 'mel_pr_auc', 'brier_score', 'ece_15',
    'unrecoverable_melanoma_count', 'unrecoverable_melanoma_rate',
]
model_table = pd.DataFrame({
    name: {metric: results[name][metric] for metric in common_metric_names}
    for name in model_names
}).T
fusion_transition_table = pd.DataFrame({
    name: {
        'offpair_to_pair_transitions': results[name]['offpair_to_pair_transitions'],
        'offpair_to_pair_transition_rate': results[name]['offpair_to_pair_transition_rate'],
    }
    for name in ['fixed_fusion', 'learned_fusion']
}).T
bootstrap_names = ['fixed_vs_auxiliary_bootstrap', 'learned_vs_auxiliary_bootstrap']
bootstrap_table = pd.DataFrame({name: results[name] for name in bootstrap_names}).T
run_table = pd.Series(results['run'], name='run_information').to_frame()
display(model_table)
display(fusion_transition_table)
display(bootstrap_table)
display(run_table)

,pair_balanced_accuracy,mel_sensitivity,mel_to_nv,nv_to_mel,macro_f1,balanced_accuracy,mel_pr_auc,brier_score,ece_15,unrecoverable_melanoma_count,unrecoverable_melanoma_rate
baseline,0.65437,0.408,54.0,22.0,0.618565,0.631745,0.579071,0.303045,0.025669,16.0,0.128
auxiliary_only,0.64963,0.400,50.0,22.0,0.613889,0.633462,0.579435,0.304334,0.033467,19.0,0.152
fixed_fusion,0.64637,0.392,51.0,21.0,0.612408,0.632531,0.583781,0.303561,0.029547,19.0,0.152
learned_fusion,0.64963,0.400,50.0,22.0,0.613103,0.633462,0.583481,0.303761,0.029738,19.0,0.152


,offpair_to_pair_transitions,offpair_to_pair_transition_rate
fixed_fusion,0.0,0.0
learned_fusion,0.0,0.0


,mean_difference,ci_95_low,ci_95_high
fixed_vs_auxiliary_bootstrap,-0.003368,-0.013158,0.0022
learned_vs_auxiliary_bootstrap,0.000000,0.000000,0.0000


,run_information
test_images,1010
mean_alpha,0.452162
device,cuda


## Outputs

The `outputs/full_20epochs` directory now contains the frozen split summary, feature and prediction caches, hourly/latest and best checkpoints for both image-level models, learning histories, learned gate, calibrated test probabilities, hardware profile, and final metrics. Keep these files to preserve resume capability and reproducibility.